# 🏆 Milestone 3 – Final Evaluation: FAISS vs Pinecone
## RAG pipeline comparison

**Goal:** Run identical queries on both pipelines → measure latency, retrieval quality, answer quality → declare a winner.

| Variable | Value |
|---|---|
| Embedding model | all-MiniLM-L6-v2 (same for both) |
| LLM | llama-3.3-70b-versatile via Groq (same for both) |
| Data source | Neo4j Knowledge Graph (same for both) |
| Top-K | 8 (same for both) |

## Cell 1 – Imports

In [1]:
# ════════════════════════════════════════════════════════════════
# Imports first — always before any usage
# ════════════════════════════════════════════════════════════════
import time
import warnings
import statistics
warnings.filterwarnings('ignore')

from dataclasses import dataclass
from typing import List, Optional

from neo4j import GraphDatabase
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# FAISS
try:
    from langchain_community.vectorstores import FAISS
    print("✅ FAISS support loaded")
except ImportError:
    print("❌ FAISS not installed — run: pip install faiss-cpu")

# Pinecone
try:
    from langchain_pinecone import PineconeVectorStore
    from pinecone import Pinecone
    print("✅ Pinecone support loaded")
except ImportError:
    print("❌ Pinecone not installed — run: pip install langchain-pinecone pinecone-client")

print("\n📅 Evaluation started:", time.strftime("%Y-%m-%d %H:%M IST"))

✅ FAISS support loaded
✅ Pinecone support loaded

📅 Evaluation started: 2026-03-27 06:27 IST


## Cell 2 – Configuration

In [ ]:
# ════════════════════════════════════════════════════════════════
# CONFIG — update credentials if needed
# ════════════════════════════════════════════════════════════════

# Neo4j
NEO4J_URI      = "your_api_key"
NEO4J_USER     = "your_username"
NEO4J_PASSWORD = "your_password"

# Groq
GROQ_API_KEY = "your_groq_api_key"
LLM_MODEL    = "llama-3.3-70b-versatile"

# Embeddings — MUST be identical for fair comparison
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

# Pinecone
PINECONE_API_KEY = "your_pinecone_api_key"
PINECONE_INDEX   = "your_index_name"

# ⚠️  FIX: matches the exact folder name saved in your FAISS notebook
FAISS_INDEX_PATH = "faiss_index_langchain"

TOP_K       = 8
TEMPERATURE = 0.2

print("✅ Configuration loaded")
print(f"   LLM            : {LLM_MODEL}")
print(f"   Embedding model: {EMBEDDING_MODEL}")
print(f"   Top-K          : {TOP_K}")
print(f"   FAISS path     : {FAISS_INDEX_PATH}")
print(f"   Pinecone index : {PINECONE_INDEX}")

✅ Configuration loaded
   LLM            : llama-3.3-70b-versatile
   Embedding model: all-MiniLM-L6-v2
   Top-K          : 8
   FAISS path     : faiss_index_langchain
   Pinecone index : job-knowledge-graph-rag


## Cell 3 – Job Dataclass + Load from Neo4j
> **Identical to both approach notebooks** — single source of truth for data loading

In [3]:
# ════════════════════════════════════════════════════════════════
# Job dataclass — exactly matches both approach notebooks
# ════════════════════════════════════════════════════════════════
@dataclass
class Job:
    job_id: str
    category: str
    workplace: str
    employment_type: str
    priority_class: str
    demand_score: float
    city: str
    country: str
    region: Optional[str] = None
    department_category: Optional[str] = None
    skills_required: List[str] = None

    def __post_init__(self):
        if self.skills_required is None:
            self.skills_required = []

    @property
    def text_description(self) -> str:
        """Exactly matches the @property in both approach notebooks"""
        parts = [
            f"Job Category: {self.category}",
            f"Workplace: {self.workplace}",
            f"Employment Type: {self.employment_type}",
            f"Priority: {self.priority_class}",
            f"Demand Score: {self.demand_score:.1f}",
            f"Location: {self.city}, {self.country}",
        ]
        if self.region:
            parts.append(f"Region: {self.region}")
        if self.department_category:
            parts.append(f"Department Category: {self.department_category}")
        if self.skills_required:
            parts.append(f"Required Skills: {', '.join(self.skills_required)}")
        return "\n".join(parts)


# ════════════════════════════════════════════════════════════════
# Load from Neo4j — identical Cypher query as both notebooks
# ════════════════════════════════════════════════════════════════
def load_jobs():
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    query = """
    MATCH (j:Job)-[:LOCATED_IN]->(l:Location)
    MATCH (j)-[:BELONGS_TO]->(c:Category)
    OPTIONAL MATCH (j)-[:IN_DEPARTMENT]->(d:Department)
    OPTIONAL MATCH (j)-[:REQUIRES]->(s:Skill)
    RETURN
        j.id                        AS job_id,
        c.name                      AS category,
        j.workplace                 AS workplace,
        j.employment_type           AS employment_type,
        j.priority_class            AS priority_class,
        toFloat(j.demand_score)     AS demand_score,
        l.city                      AS city,
        l.country                   AS country,
        l.region                    AS region,
        d.category                  AS department_category,
        collect(DISTINCT s.name)    AS skills_required
    """
    jobs = []
    with driver.session() as session:
        for r in session.run(query):
            jobs.append(Job(
                job_id              = str(r["job_id"]),
                category            = r["category"],
                workplace           = r["workplace"],
                employment_type     = r["employment_type"],
                priority_class      = r["priority_class"],
                demand_score        = r["demand_score"],
                city                = r["city"] or "Unknown",
                country             = r["country"] or "Unknown",
                region              = r["region"],
                department_category = r["department_category"],
                skills_required     = r["skills_required"] or []
            ))
    driver.close()
    return jobs


print("⏳ Loading jobs from Neo4j...")
jobs = load_jobs()
print(f"✅ Loaded {len(jobs)} jobs")
print(f"   Jobs with skills : {sum(1 for j in jobs if j.skills_required)}")
print(f"   Unique countries : {len(set(j.country for j in jobs))}")
print(f"   Unique categories: {len(set(j.category for j in jobs))}")
print(f"\n📄 Sample text_description (first job):")
print(jobs[0].text_description)

⏳ Loading jobs from Neo4j...
✅ Loaded 644 jobs
   Jobs with skills : 644
   Unique countries : 62
   Unique categories: 6

📄 Sample text_description (first job):
Job Category: Business Analyst
Workplace: Remote
Employment Type: Full-Time
Priority: Normal-High
Demand Score: 26.9
Location: Unknown, United Kingdom
Region: Europe
Department Category: Operations
Required Skills: Excel, SQL


## Cell 4 – Build LangChain Documents (shared by both pipelines)

In [4]:
# ════════════════════════════════════════════════════════════════
# Same documents fed into both FAISS and Pinecone
# ════════════════════════════════════════════════════════════════
documents = []
for job in jobs:
    doc = Document(
        page_content=job.text_description,
        metadata={
            "job_id"             : job.job_id,
            "category"           : job.category,
            "workplace"          : job.workplace,
            "employment_type"    : job.employment_type,
            "priority_class"     : job.priority_class,
            "demand_score"       : job.demand_score,
            "city"               : job.city,
            "country"            : job.country,
            "region"             : job.region or "",
            "department_category": job.department_category or "",
            "skills"             : ", ".join(job.skills_required)
        }
    )
    documents.append(doc)

print(f"✅ {len(documents)} LangChain Documents ready")
print(f"   page_content sample:")
print("   " + documents[0].page_content.replace("\n", "\n   "))

✅ 644 LangChain Documents ready
   page_content sample:
   Job Category: Business Analyst
   Workplace: Remote
   Employment Type: Full-Time
   Priority: Normal-High
   Demand Score: 26.9
   Location: Unknown, United Kingdom
   Region: Europe
   Department Category: Operations
   Required Skills: Excel, SQL


## Cell 5 – Load Embedding Model (shared)

In [5]:
# ════════════════════════════════════════════════════════════════
# Single embedding model instance — used by BOTH pipelines
# This guarantees vectors are computed identically
# ════════════════════════════════════════════════════════════════
print("⏳ Loading embedding model (may take 1-2 min first time)...")
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)
print(f"✅ Embedding model loaded: {EMBEDDING_MODEL}")

⏳ Loading embedding model (may take 1-2 min first time)...


C:\Users\Anushree\AppData\Local\Temp\ipykernel_31876\1215797831.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model loaded: all-MiniLM-L6-v2


## Cell 6 – FAISS Pipeline Setup

In [6]:
# ════════════════════════════════════════════════════════════════
# FAISS — try loading saved index first, build if not found
# ════════════════════════════════════════════════════════════════
print(f"⏳ Setting up FAISS pipeline...")
print(f"   Looking for saved index at: '{FAISS_INDEX_PATH}'")

try:
    faiss_store = FAISS.load_local(
        FAISS_INDEX_PATH,
        embeddings,
        allow_dangerous_deserialization=True
    )
    print(f"✅ FAISS index loaded from disk ({faiss_store.index.ntotal} vectors)")
except Exception as e:
    print(f"   Index not found on disk ({e})")
    print("   Building fresh FAISS index from documents...")
    faiss_store = FAISS.from_documents(documents, embeddings)
    faiss_store.save_local(FAISS_INDEX_PATH)
    print(f"✅ FAISS index built & saved ({faiss_store.index.ntotal} vectors)")

# Retriever — plain similarity (same as Pinecone for fair comparison)
faiss_retriever = faiss_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": TOP_K}
)

# LLM
faiss_llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model=LLM_MODEL,
    temperature=TEMPERATURE,
    max_tokens=600
)

# Format docs function — named fn, not lambda (avoids LangChain serialization issues)
def faiss_format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Prompt — identical to Pinecone
EVAL_PROMPT = PromptTemplate.from_template(
    """You are a job search assistant. Answer using ONLY the job listings below.
Do NOT invent information.

Retrieved Jobs:
{context}

Question: {question}

Answer (mention job count, key patterns, locations, skills):"""
)

faiss_chain = (
    {"context": faiss_retriever | faiss_format_docs, "question": RunnablePassthrough()}
    | EVAL_PROMPT
    | faiss_llm
    | StrOutputParser()
)

print("✅ FAISS pipeline ready")

⏳ Setting up FAISS pipeline...
   Looking for saved index at: 'faiss_index_langchain'
✅ FAISS index loaded from disk (644 vectors)
✅ FAISS pipeline ready


## Cell 7 – Pinecone Pipeline Setup

In [7]:
# ════════════════════════════════════════════════════════════════
# Pinecone — connect to existing cloud index
# ════════════════════════════════════════════════════════════════
print("⏳ Connecting to Pinecone...")

pc = Pinecone(api_key=PINECONE_API_KEY)

# ⚠️  Sanity check — confirm index has vectors before running any queries
index_handle = pc.Index(PINECONE_INDEX)
stats = index_handle.describe_index_stats()
vector_count = stats.get('total_vector_count', 0)
print(f"   Pinecone index '{PINECONE_INDEX}' → {vector_count} vectors")

if vector_count == 0:
    raise RuntimeError(
        "Pinecone index is empty! "
        "Re-run your Pinecone notebook to upsert documents first."
    )

pinecone_store = PineconeVectorStore(
    index_name=PINECONE_INDEX,
    embedding=embeddings,           # same embedding instance as FAISS
    pinecone_api_key=PINECONE_API_KEY
)

pinecone_retriever = pinecone_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": TOP_K}
)

pinecone_llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model=LLM_MODEL,
    temperature=TEMPERATURE,
    max_tokens=600
)

def pinecone_format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Same prompt as FAISS — controlled comparison
pinecone_chain = (
    {"context": pinecone_retriever | pinecone_format_docs, "question": RunnablePassthrough()}
    | EVAL_PROMPT
    | pinecone_llm
    | StrOutputParser()
)

print("✅ Pinecone pipeline ready")

⏳ Connecting to Pinecone...
   Pinecone index 'job-knowledge-graph-rag' → 644 vectors
✅ Pinecone pipeline ready


## Cell 8 – Evaluation Function
> Measures **retrieval latency** and **total latency** separately — this is the key fix.
> Retrieval latency is where FAISS vs Pinecone actually differ. Total latency is dominated by the LLM and will look similar for both.

In [8]:
# ════════════════════════════════════════════════════════════════
# Evaluation function
# Measures: retrieval latency, total latency, docs retrieved, answer
# ════════════════════════════════════════════════════════════════
def evaluate_query(query: str, query_num: int, total: int):
    print(f"\n{'═'*80}")
    print(f"Query {query_num}/{total}: {query}")
    print('═'*80)

    result = {"query": query}

    # ── FAISS ────────────────────────────────────────────────────
    # Retrieval latency (vector search only)
    try:
        t0 = time.time()
        faiss_docs = faiss_retriever.invoke(query)
        result["faiss_retrieval_ms"] = round((time.time() - t0) * 1000, 1)

        # Total latency (retrieval + LLM)
        t1 = time.time()
        result["faiss_answer"] = faiss_chain.invoke(query)
        result["faiss_total_s"] = round(time.time() - t1, 2)

        result["faiss_docs_count"] = len(faiss_docs)
        result["faiss_error"] = None

        print(f"\n🔵 FAISS")
        print(f"   Retrieval : {result['faiss_retrieval_ms']} ms")
        print(f"   Total     : {result['faiss_total_s']} s")
        print(f"   Docs found: {result['faiss_docs_count']}")
        print(f"   Answer    : {result['faiss_answer'].strip()[:350]}")

    except Exception as e:
        result["faiss_answer"] = f"ERROR: {e}"
        result["faiss_retrieval_ms"] = None
        result["faiss_total_s"] = None
        result["faiss_docs_count"] = 0
        result["faiss_error"] = str(e)
        print(f"\n🔵 FAISS ERROR: {e}")

    print("-"*80)

    # ── Pinecone ─────────────────────────────────────────────────
    try:
        t0 = time.time()
        pine_docs = pinecone_retriever.invoke(query)
        result["pine_retrieval_ms"] = round((time.time() - t0) * 1000, 1)

        t1 = time.time()
        result["pine_answer"] = pinecone_chain.invoke(query)
        result["pine_total_s"] = round(time.time() - t1, 2)

        result["pine_docs_count"] = len(pine_docs)
        result["pine_error"] = None

        print(f"\n🟠 Pinecone")
        print(f"   Retrieval : {result['pine_retrieval_ms']} ms")
        print(f"   Total     : {result['pine_total_s']} s")
        print(f"   Docs found: {result['pine_docs_count']}")
        print(f"   Answer    : {result['pine_answer'].strip()[:350]}")

    except Exception as e:
        result["pine_answer"] = f"ERROR: {e}"
        result["pine_retrieval_ms"] = None
        result["pine_total_s"] = None
        result["pine_docs_count"] = 0
        result["pine_error"] = str(e)
        print(f"\n🟠 Pinecone ERROR: {e}")

    # ── Per-query delta ──────────────────────────────────────────
    if result.get("faiss_retrieval_ms") and result.get("pine_retrieval_ms"):
        delta = result["pine_retrieval_ms"] - result["faiss_retrieval_ms"]
        faster = "FAISS" if delta > 0 else "Pinecone"
        print(f"\n   ⚡ Retrieval delta: {abs(delta):.1f} ms faster → {faster}")

    print('═'*80)
    return result

print("✅ Evaluation function ready")

✅ Evaluation function ready


## Cell 9 – Run All Queries

In [9]:
# ════════════════════════════════════════════════════════════════
# 8 evaluation queries — designed to test different aspects:
# skills match, location match, category match, combined filters
# ════════════════════════════════════════════════════════════════
queries = [
    "remote business analyst jobs requiring SQL",
    "data scientist roles with Python skills",
    "high demand UI/UX designer positions",
    "cloud engineering jobs requiring AWS",
    "hybrid software developer jobs in UK",
    "jobs in Europe requiring Agile methodology",
    "remote jobs needing Excel and PowerPoint",
    "top skills required for business analyst roles"
]

results = []
total = len(queries)

for i, q in enumerate(queries, 1):
    result = evaluate_query(q, i, total)
    results.append(result)
    if i < total:
        time.sleep(1.5)  # Groq rate limit buffer

print(f"\n✅ All {total} queries completed.")


════════════════════════════════════════════════════════════════════════════════
Query 1/8: remote business analyst jobs requiring SQL
════════════════════════════════════════════════════════════════════════════════

🔵 FAISS
   Retrieval : 88.6 ms
   Total     : 0.66 s
   Docs found: 8
   Answer    : There are 8 remote Business Analyst jobs requiring SQL. 
Key patterns include the requirement of SQL, Data Visualization, and Excel skills in most jobs. 
Locations are varied, including the United States (Boston, Cherry Hill, Washington, and Unknown), the United Kingdom (London), India (Delhi), and Jordan (Unknown). 
Common required skills across t
--------------------------------------------------------------------------------

🟠 Pinecone
   Retrieval : 2467.9 ms
   Total     : 0.77 s
   Docs found: 8
   Answer    : There are 8 remote Business Analyst jobs requiring SQL. 
Key patterns include the requirement of SQL and Data Visualization skills in most jobs, and the prevalence of remote 

## Cell 10 – Performance Summary Table

In [10]:
# ════════════════════════════════════════════════════════════════
# Structured summary table — latency per query
# ════════════════════════════════════════════════════════════════
print("\n" + "═"*90)
print("  PERFORMANCE SUMMARY")
print("═"*90)
print(f"{'#':<3} {'Query':<42} {'FAISS retr':>10} {'Pine retr':>10} {'FAISS total':>12} {'Pine total':>11}")
print("-"*90)

faiss_retrieval_times = []
pine_retrieval_times  = []
faiss_total_times     = []
pine_total_times      = []
faiss_wins = 0
pine_wins  = 0

for i, r in enumerate(results, 1):
    fr = f"{r['faiss_retrieval_ms']} ms" if r.get('faiss_retrieval_ms') else "ERR"
    pr = f"{r['pine_retrieval_ms']} ms"  if r.get('pine_retrieval_ms')  else "ERR"
    ft = f"{r['faiss_total_s']} s"       if r.get('faiss_total_s')      else "ERR"
    pt = f"{r['pine_total_s']} s"        if r.get('pine_total_s')       else "ERR"

    print(f"{i:<3} {r['query'][:40]:<42} {fr:>10} {pr:>10} {ft:>12} {pt:>11}")

    if r.get('faiss_retrieval_ms'): faiss_retrieval_times.append(r['faiss_retrieval_ms'])
    if r.get('pine_retrieval_ms'):  pine_retrieval_times.append(r['pine_retrieval_ms'])
    if r.get('faiss_total_s'):      faiss_total_times.append(r['faiss_total_s'])
    if r.get('pine_total_s'):       pine_total_times.append(r['pine_total_s'])

    if r.get('faiss_retrieval_ms') and r.get('pine_retrieval_ms'):
        if r['faiss_retrieval_ms'] <= r['pine_retrieval_ms']:
            faiss_wins += 1
        else:
            pine_wins += 1

print("-"*90)

if faiss_retrieval_times:
    avg_fr = round(statistics.mean(faiss_retrieval_times), 1)
    avg_ft = round(statistics.mean(faiss_total_times), 2)
else:
    avg_fr = avg_ft = "N/A"

if pine_retrieval_times:
    avg_pr = round(statistics.mean(pine_retrieval_times), 1)
    avg_pt = round(statistics.mean(pine_total_times), 2)
else:
    avg_pr = avg_pt = "N/A"

print(f"{'AVG':<3} {'':42} {str(avg_fr)+' ms':>10} {str(avg_pr)+' ms':>10} {str(avg_ft)+' s':>12} {str(avg_pt)+' s':>11}")
print(f"{'WIN':<3} {'(retrieval latency)':42} {f'🔵 {faiss_wins}':>10} {f'🟠 {pine_wins}':>10}")
print("═"*90)


══════════════════════════════════════════════════════════════════════════════════════════
  PERFORMANCE SUMMARY
══════════════════════════════════════════════════════════════════════════════════════════
#   Query                                      FAISS retr  Pine retr  FAISS total  Pine total
------------------------------------------------------------------------------------------
1   remote business analyst jobs requiring S      88.6 ms  2467.9 ms       0.66 s      0.77 s
2   data scientist roles with Python skills       21.3 ms   326.6 ms       0.64 s      0.87 s
3   high demand UI/UX designer positions          24.9 ms   301.1 ms       0.44 s      0.91 s
4   cloud engineering jobs requiring AWS          12.7 ms   337.4 ms       0.73 s      0.89 s
5   hybrid software developer jobs in UK          19.3 ms   290.2 ms       0.54 s      0.91 s
6   jobs in Europe requiring Agile methodolo      15.7 ms   291.1 ms       0.63 s      0.82 s
7   remote jobs needing Excel and PowerPoint  

## Cell 11 – Final Verdict

In [ ]:
# ════════════════════════════════════════════════════════════════
# FINAL VERDICT — data-driven, no fluff
# ════════════════════════════════════════════════════════════════
print("\n" + "═"*70)
print("  FINAL VERDICT")
print("═"*70)

# ── Latency winner ──
if faiss_retrieval_times and pine_retrieval_times:
    avg_fr = statistics.mean(faiss_retrieval_times)
    avg_pr = statistics.mean(pine_retrieval_times)
    latency_winner = "FAISS" if avg_fr < avg_pr else "Pinecone"
    latency_margin = abs(avg_pr - avg_fr)
    print(f"\n⚡ Retrieval Speed")
    print(f"   FAISS avg    : {avg_fr:.1f} ms")
    print(f"   Pinecone avg : {avg_pr:.1f} ms")
    print(f"   Winner       : {latency_winner} (faster by {latency_margin:.1f} ms avg)")

# ── Win count ──
total_comparable = faiss_wins + pine_wins
if total_comparable > 0:
    print(f"\n🏅 Per-query wins (retrieval speed)")
    print(f"   FAISS    : {faiss_wins}/{total_comparable} queries")
    print(f"   Pinecone : {pine_wins}/{total_comparable} queries")

# ── Data scale context ──
print(f"\n📦 Data scale")
print(f"   Total jobs in graph : {len(jobs)}")
print(f"   Vectors indexed     : {len(documents)}")
if faiss_retrieval_times:
    print(f"   At this scale ({len(jobs)} docs), both stores are fast.")
    print(f"   The gap between them is {'negligible (<5ms)' if latency_margin < 5 else f'meaningful ({latency_margin:.0f}ms avg)'}")

# ── Overall winner ──
print(f"\n" + "─"*70)
print(f"  VERDICT FOR THIS PROJECT")
print(f"─"*70)

if faiss_wins >= pine_wins and avg_fr <= avg_pr:
    winner = "FAISS"
    reason = (
        f"FAISS wins on retrieval speed ({avg_fr:.1f}ms vs {avg_pr:.1f}ms) "
        f"and won {faiss_wins}/{total_comparable} head-to-head query comparisons. "
        f"At {len(jobs)} documents, the dataset fits entirely in memory — "
        f"FAISS's in-memory flat index is the natural fit. "
        f"No network round-trip, no API dependency, no cost. "
        f"For Milestone 4's semantic layer, go with FAISS."
    )
else:
    winner = "Pinecone"
    reason = (
        f"Pinecone wins on retrieval speed ({avg_pr:.1f}ms vs {avg_fr:.1f}ms) "
        f"and won {pine_wins}/{total_comparable} head-to-head query comparisons. "
        f"The persistent cloud index means no rebuild on every session, "
        f"and results are consistent regardless of environment. "
        f"For Milestone 4's semantic layer, go with Pinecone."
    )

print(f"\n  ✅ GO WITH: {winner}")
print(f"\n  Why: {reason}")
print("═"*70)
print(f"\n  Evaluation completed: {time.strftime('%Y-%m-%d %H:%M IST')}")
print("═"*70)



══════════════════════════════════════════════════════════════════════
  FINAL VERDICT
══════════════════════════════════════════════════════════════════════

⚡ Retrieval Speed
   FAISS avg    : 27.7 ms
   Pinecone avg : 581.6 ms
   Winner       : FAISS (faster by 553.9 ms avg)

🏅 Per-query wins (retrieval speed)
   FAISS    : 8/8 queries
   Pinecone : 0/8 queries

📦 Data scale
   Total jobs in graph : 644
   Vectors indexed     : 644
   At this scale (644 docs), both stores are fast.
   The gap between them is meaningful (554ms avg)

──────────────────────────────────────────────────────────────────────
  VERDICT FOR THIS PROJECT
──────────────────────────────────────────────────────────────────────

  ✅ GO WITH: FAISS

  Why: FAISS wins on retrieval speed (27.7ms vs 581.6ms) and won 8/8 head-to-head query comparisons. At 644 documents, the dataset fits entirely in memory — FAISS's in-memory flat index is the natural fit. No network round-trip, no API dependency, no cost. For Milesto